# 03 - Analisis exploratorio orientado a storytelling e insights

Este notebook transforma el EDA tecnico en un analisis narrativo, orientado a comunicar hallazgos sobre siniestros viales en CABA. El foco no esta en modelar ni predecir, sino en construir evidencia exploratoria, formular interpretaciones razonables y explicitar limitaciones.

El trabajo utiliza el dataset procesado `data/processed/siniestros_limpio.csv`. No se modifica ningun archivo en `data/raw/`.

## 1. Introduccion

Los siniestros viales son eventos de alto impacto social porque afectan la movilidad, la salud publica y la planificacion urbana. Un analisis exploratorio avanzado no debe limitarse a contar registros: debe conectar los datos con preguntas de dominio, interpretar categorias segun su definicion institucional y construir una narrativa que ayude a comprender donde aparecen los mayores niveles de severidad.

En este notebook se prioriza una lectura orientada a storytelling. Cada analisis se organiza a partir de una pregunta analitica, una hipotesis, visualizaciones, interpretacion escrita y limitaciones. Esta estructura permite separar lo que los datos muestran de aquello que solo puede proponerse como interpretacion preliminar.

## 2. Contexto del dataset

El dataset corresponde a victimas de siniestros viales en CABA y proviene de un archivo oficial que incluye una hoja de diccionario de datos. Esa metadata es central para el analisis porque define el significado institucional de las variables y evita tratar las categorias como strings arbitrarios.

Aspectos semanticos clave:

- `SD` significa **Sin Datos**. Debe interpretarse como ausencia de informacion y no como una categoria sustantiva del fenomeno vial.
- `gravedad_victima` es una variable ordinal: `LEVE < GRAVE < MORTAL`.
- `LEVE` refiere a alta medica dentro de las 24 horas o hechos sin datos sobre gravedad de lesiones.
- `GRAVE` refiere a hospitalizacion de al menos 24 horas o atencion especializada.
- `MORTAL` refiere a fallecimiento dentro de los 30 dias posteriores al siniestro por causas atribuibles al hecho.

Estas definiciones condicionan la lectura de resultados. Por ejemplo, una comparacion entre modos de desplazamiento debe considerar simultaneamente volumen de casos, proporcion de severidad y calidad de registro.

## 3. Preguntas analiticas

- Como varia la gravedad de las victimas segun el modo de desplazamiento?
- Que modos presentan mayor proporcion relativa de victimas graves o mortales?
- Los modos con mayor cantidad de victimas son tambien los que muestran mayor severidad relativa?
- Que categorias requieren cautela por presencia de `SD` o bajo volumen de casos?
- Que patrones iniciales pueden orientar futuras preguntas de investigacion?

## 4. Hipotesis

1. Los modos de desplazamiento con menor proteccion fisica podrian concentrar mayor proporcion relativa de victimas graves o mortales.
2. Los modos mas frecuentes podrian dominar los conteos absolutos sin ser necesariamente los mas severos en terminos porcentuales.
3. Las categorias residuales o con pocos registros pueden generar porcentajes inestables y deben interpretarse con cuidado.
4. La presencia de `SD` puede revelar problemas de completitud y afectar la lectura sustantiva de los resultados.

## Preparacion del entorno

Se utilizan exclusivamente `pandas`, `matplotlib` y `seaborn` para el analisis exploratorio. Tambien se importa la metadata documentada en `src.data_loader` para respetar el orden ordinal de `gravedad_victima` y la interpretacion de `SD`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_loader import (
    GRAVEDAD_VICTIMA_ORDER,
    OFFICIAL_CATEGORY_DEFINITIONS,
    SD_MEANING,
    SD_VALUE,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

sns.set_theme(style="whitegrid", palette="deep")

## Carga del dataset procesado

El analisis parte del archivo procesado preliminar. La celda siguiente verifica su existencia y lo carga en memoria. No se lee ni se sobrescribe ningun archivo raw.

In [ ]:
DATA_FILE = PROJECT_ROOT / "data" / "processed" / "siniestros_limpio.csv"

if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"No se encontro el dataset procesado esperado: {DATA_FILE}. "
        "Ejecute primero notebooks/02_preprocessing.ipynb."
    )

df = pd.read_csv(DATA_FILE)

print(f"Dataset procesado cargado desde: {DATA_FILE.resolve()}")
print(f"Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas")
display(df.head())

## Preparacion analitica

Se crea una copia de trabajo para el analisis. El objetivo es estandarizar nombres y categorias sin alterar el archivo procesado. La variable de gravedad se ordena segun el diccionario oficial.

In [ ]:
df_analisis = df.copy()

if "gravedad_victima" not in df_analisis.columns and "GRAVEdad_victima" in df_analisis.columns:
    df_analisis = df_analisis.rename(columns={"GRAVEdad_victima": "gravedad_victima"})

required_columns = ["modo_desplazamiento_victima", "gravedad_victima"]
missing_columns = [column for column in required_columns if column not in df_analisis.columns]
if missing_columns:
    raise KeyError(f"Faltan columnas requeridas para el analisis: {missing_columns}")

df_analisis["modo_desplazamiento_victima"] = (
    df_analisis["modo_desplazamiento_victima"]
    .astype("string")
    .str.strip()
    .str.upper()
)

df_analisis["gravedad_victima"] = (
    df_analisis["gravedad_victima"]
    .astype("string")
    .str.strip()
    .str.upper()
)

orden_gravedad = list(GRAVEDAD_VICTIMA_ORDER.keys())
df_analisis["gravedad_victima"] = pd.Categorical(
    df_analisis["gravedad_victima"],
    categories=orden_gravedad,
    ordered=True,
)

definiciones_gravedad = pd.DataFrame({
    "categoria": orden_gravedad,
    "orden": [GRAVEDAD_VICTIMA_ORDER[categoria] for categoria in orden_gravedad],
    "definicion_institucional": [
        OFFICIAL_CATEGORY_DEFINITIONS["gravedad_victima"][categoria]
        for categoria in orden_gravedad
    ],
})

display(definiciones_gravedad)

## 5. Visualizaciones

### Analisis 1: Gravedad de victimas segun modo de desplazamiento

**Pregunta analitica.** Como se distribuye la gravedad de las victimas segun el modo de desplazamiento?

**Hipotesis.** Los modos de desplazamiento con menor proteccion fisica podrian presentar una mayor proporcion relativa de victimas graves o mortales, aunque los modos mas frecuentes pueden dominar los conteos absolutos.

**Relevancia.** Este cruce es importante porque conecta una consecuencia del siniestro, la severidad de la victima, con una condicion de exposicion vial. En terminos de comunicacion de hallazgos, permite distinguir entre magnitud del problema y composicion de gravedad.

#### Tabla porcentual

La tabla porcentual responde: de cada 100 victimas registradas en un modo de desplazamiento, que porcentaje corresponde a lesiones leves, graves o mortales? Esta mirada facilita comparaciones entre modos con volumenes distintos.

In [ ]:
tabla_frecuencias = pd.crosstab(
    df_analisis["modo_desplazamiento_victima"],
    df_analisis["gravedad_victima"],
    dropna=False,
).reindex(columns=orden_gravedad, fill_value=0)

tabla_frecuencias["TOTAL"] = tabla_frecuencias.sum(axis=1)
tabla_frecuencias = tabla_frecuencias.sort_values("TOTAL", ascending=False)

tabla_porcentual = (
    tabla_frecuencias[orden_gravedad]
    .div(tabla_frecuencias["TOTAL"].replace(0, pd.NA), axis=0)
    .mul(100)
    .round(2)
)

tabla_porcentual["TOTAL_REGISTROS"] = tabla_frecuencias["TOTAL"]

display(tabla_porcentual)

#### Grafico de barras apiladas

El grafico apilado porcentual permite comparar la composicion interna de gravedad entre modos. Al usar porcentajes, cada barra representa el 100% de las victimas de ese modo; esto evita que los modos con mas registros oculten patrones relativos de severidad.

In [ ]:
plot_porcentual = tabla_porcentual[orden_gravedad].copy()

ax = plot_porcentual.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6),
    color=["#4C78A8", "#F58518", "#E45756"],
)

ax.set_title("Composicion porcentual de gravedad segun modo de desplazamiento")
ax.set_xlabel("Modo de desplazamiento de la victima")
ax.set_ylabel("Porcentaje dentro del modo")
ax.legend(title="Gravedad", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_ylim(0, 100)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

#### Heatmap porcentual

El heatmap ayuda a localizar rapidamente concentraciones relativas. Los tonos mas intensos indican mayor porcentaje dentro de cada modo de desplazamiento, no mayor cantidad absoluta de casos.

In [ ]:
plt.figure(figsize=(9, max(4, 0.45 * len(tabla_porcentual))))
sns.heatmap(
    tabla_porcentual[orden_gravedad],
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.5,
    cbar_kws={"label": "Porcentaje dentro del modo"},
)
plt.title("Heatmap porcentual de gravedad por modo de desplazamiento")
plt.xlabel("Gravedad de la victima")
plt.ylabel("Modo de desplazamiento de la victima")
plt.tight_layout()
plt.show()

## 6. Insights escritos

### Insights automaticos

La siguiente celda genera conclusiones descriptivas a partir de la tabla porcentual. Estos resultados no reemplazan la interpretacion humana: funcionan como una primera sintesis reproducible para orientar el relato analitico.

In [ ]:
metricas_modo = tabla_frecuencias.copy()
metricas_modo["pct_leve"] = tabla_porcentual["LEVE"]
metricas_modo["pct_grave"] = tabla_porcentual["GRAVE"]
metricas_modo["pct_mortal"] = tabla_porcentual["MORTAL"]
metricas_modo["pct_grave_mortal"] = (
    tabla_porcentual[["GRAVE", "MORTAL"]].sum(axis=1).round(2)
)

incluye_sd = SD_VALUE in metricas_modo.index
metricas_sustantivas = metricas_modo.drop(index=SD_VALUE, errors="ignore")

modo_mas_frecuente = metricas_modo["TOTAL"].idxmax()
total_mas_frecuente = int(metricas_modo.loc[modo_mas_frecuente, "TOTAL"])

modo_sustantivo_mas_frecuente = metricas_sustantivas["TOTAL"].idxmax()
total_sustantivo_mas_frecuente = int(
    metricas_sustantivas.loc[modo_sustantivo_mas_frecuente, "TOTAL"]
)

modo_mayor_grave_mortal = metricas_sustantivas["pct_grave_mortal"].idxmax()
pct_mayor_grave_mortal = float(
    metricas_sustantivas.loc[modo_mayor_grave_mortal, "pct_grave_mortal"]
)

modo_mayor_mortal = metricas_sustantivas["pct_mortal"].idxmax()
pct_mayor_mortal = float(metricas_sustantivas.loc[modo_mayor_mortal, "pct_mortal"])

modos_bajo_volumen = metricas_sustantivas[metricas_sustantivas["TOTAL"] < 30].index.tolist()

conclusiones_automaticas = [
    f"Incluyendo todas las categorias, el mayor volumen aparece en {modo_mas_frecuente}, con {total_mas_frecuente} registros.",
    f"Excluyendo {SD_VALUE}, el modo sustantivo con mayor volumen es {modo_sustantivo_mas_frecuente}, con {total_sustantivo_mas_frecuente} registros.",
    f"Entre modos sustantivos, la mayor proporcion combinada de victimas GRAVE o MORTAL aparece en {modo_mayor_grave_mortal}, con {pct_mayor_grave_mortal:.2f}% dentro de ese modo.",
    f"Entre modos sustantivos, la mayor proporcion de victimas MORTAL aparece en {modo_mayor_mortal}, con {pct_mayor_mortal:.2f}% dentro de ese modo.",
]

if modos_bajo_volumen:
    conclusiones_automaticas.append(
        "Los siguientes modos sustantivos tienen bajo volumen de registros y sus porcentajes pueden ser inestables: "
        + ", ".join(modos_bajo_volumen)
        + "."
    )

if incluye_sd:
    conclusiones_automaticas.append(
        f"La categoria {SD_VALUE} debe interpretarse como {SD_MEANING}; informa calidad de datos y no un modo de desplazamiento sustantivo."
    )

for conclusion in conclusiones_automaticas:
    print(f"- {conclusion}")

### Interpretacion escrita

La lectura de este analisis debe separar dos dimensiones. La primera es la magnitud: cuantos registros aparecen en cada modo de desplazamiento. La segunda es la severidad relativa: que proporcion de esos registros corresponde a victimas `GRAVE` o `MORTAL`. Un modo puede ser dominante en volumen por su presencia en la movilidad cotidiana, pero no necesariamente ser el que concentra mayor severidad porcentual.

El grafico de barras apiladas facilita comparar composicion, mientras que el heatmap permite detectar rapidamente celdas de mayor intensidad porcentual. Si aparecen modos con porcentajes altos de `MORTAL` o de `GRAVE + MORTAL`, esos casos deben ser considerados senales exploratorias para profundizar. No constituyen por si mismos evidencia causal, pero si orientan nuevas preguntas sobre vulnerabilidad, infraestructura, exposicion y condiciones del hecho.

Tambien es importante interpretar `SD` como una marca de falta de informacion. Su presencia no describe un tipo de usuario vial, sino una limitacion en la calidad o completitud del registro.

### Limitaciones del analisis

- El analisis es descriptivo y no establece causalidad.
- Los porcentajes pueden ser inestables en modos con bajo volumen de registros.
- La categoria `SD` representa ausencia de datos y no debe compararse sustantivamente con modos reales.
- `LEVE` incluye hechos sin datos sobre la gravedad de lesiones segun la definicion institucional, lo que puede afectar la interpretacion de baja severidad.
- No se controla por exposicion, cantidad de viajes, zona, horario, edad, rol de la victima ni otras variables contextuales.
- El dataset procesado es preliminar, por lo que futuros ajustes de limpieza pueden modificar marginalmente los resultados.

## Analisis 2: Edad de la victima vs gravedad

### 1. Pregunta analitica

Existe una relacion observable entre la edad de la victima y la severidad de las lesiones registradas?

Esta pregunta es relevante porque la edad puede modificar la vulnerabilidad fisica ante un siniestro vial. Personas de mayor edad pueden presentar mayor fragilidad biologica, tiempos de recuperacion mas prolongados o mayor probabilidad de complicaciones ante lesiones similares. Al mismo tiempo, la edad tambien puede estar asociada a patrones de movilidad, modos de desplazamiento y exposicion diferencial al riesgo.

### 2. Hipotesis

La hipotesis exploratoria es que los grupos de mayor gravedad (`GRAVE` y `MORTAL`) podrian presentar edades centrales mas altas que el grupo `LEVE`. Esto no implica causalidad: una diferencia en edad puede estar mediada por otros factores, como modo de desplazamiento, rol de la victima, zona del siniestro, horario, velocidad, infraestructura o condiciones de salud previas no observadas en el dataset.

### Preparacion de edad valida

Se trabaja unicamente con registros que poseen `edad_victima` valida. Los valores `SD`, vacios o no numericos se convierten a valores faltantes dentro de una copia de analisis y luego se excluyen de este bloque. No se modifica el dataset procesado en disco.

In [ ]:
df_edad = df_analisis.copy()

if "edad_victima" not in df_edad.columns:
    raise KeyError("No se encontro la columna edad_victima en el dataset procesado.")

df_edad["edad_victima_num"] = pd.to_numeric(
    df_edad["edad_victima"].replace(SD_VALUE, pd.NA),
    errors="coerce",
)

df_edad_valida = df_edad.dropna(subset=["edad_victima_num", "gravedad_victima"]).copy()
df_edad_valida = df_edad_valida[df_edad_valida["edad_victima_num"].between(0, 120)]

print(f"Registros totales: {len(df_edad):,}")
print(f"Registros con edad valida para este analisis: {len(df_edad_valida):,}")
print(f"Registros excluidos por edad faltante, no numerica o fuera de rango: {len(df_edad) - len(df_edad_valida):,}")

### 3. Estadisticas descriptivas

Las estadisticas descriptivas permiten comparar tendencia central y dispersion de edad entre niveles de gravedad. Se reportan promedio, mediana, desvio estandar, minimo y maximo por categoria ordinal de severidad.

In [ ]:
estadisticas_edad_gravedad = (
    df_edad_valida
    .groupby("gravedad_victima", observed=False)["edad_victima_num"]
    .agg(
        cantidad="count",
        promedio="mean",
        mediana="median",
        desvio_estandar="std",
        minimo="min",
        maximo="max",
    )
    .reindex(orden_gravedad)
    .round(2)
)

display(estadisticas_edad_gravedad)

### 4. Visualizaciones

#### Boxplot: edad de la victima vs gravedad

El boxplot resume mediana, rango intercuartil y valores extremos. Es util para comparar si la edad central y la dispersion cambian entre `LEVE`, `GRAVE` y `MORTAL`.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df_edad_valida,
    x="gravedad_victima",
    y="edad_victima_num",
    order=orden_gravedad,
    color="#8E6C8A",
)
plt.title("Edad de la victima segun gravedad de la lesion")
plt.xlabel("Gravedad de la victima")
plt.ylabel("Edad de la victima")
plt.tight_layout()
plt.show()

#### Violinplot: distribucion de edades por gravedad

El violinplot permite observar la forma de la distribucion. A diferencia del boxplot, muestra donde se concentran las edades dentro de cada nivel de gravedad y si existen distribuciones asimetricas o multimodales.

In [ ]:
plt.figure(figsize=(8, 5))
sns.violinplot(
    data=df_edad_valida,
    x="gravedad_victima",
    y="edad_victima_num",
    hue="gravedad_victima",
    order=orden_gravedad,
    hue_order=orden_gravedad,
    inner="quartile",
    cut=0,
    palette=["#4C78A8", "#F58518", "#E45756"],
    legend=False,
)
plt.title("Distribucion de edad por gravedad de la victima")
plt.xlabel("Gravedad de la victima")
plt.ylabel("Edad de la victima")
plt.tight_layout()
plt.show()

#### Histograma superpuesto

El histograma superpuesto es una visualizacion complementaria. Ayuda a comparar la presencia relativa de edades en cada grupo de gravedad, aunque puede volverse mas dificil de leer cuando las clases estan desbalanceadas.

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(
    data=df_edad_valida,
    x="edad_victima_num",
    hue="gravedad_victima",
    hue_order=orden_gravedad,
    bins=30,
    stat="density",
    common_norm=False,
    element="step",
    fill=False,
    palette=["#4C78A8", "#F58518", "#E45756"],
)
plt.title("Histograma superpuesto de edad por gravedad")
plt.xlabel("Edad de la victima")
plt.ylabel("Densidad")
plt.tight_layout()
plt.show()

### 5. Insights escritos

La siguiente celda genera una interpretacion automatica sobre edades centrales, dispersion y posibles patrones de vulnerabilidad etaria. Los resultados son descriptivos y deben leerse como senales exploratorias.

In [ ]:
estadisticas_validas = estadisticas_edad_gravedad.dropna(subset=["promedio", "mediana"])

grupo_promedio_mayor = estadisticas_validas["promedio"].idxmax()
promedio_mayor = float(estadisticas_validas.loc[grupo_promedio_mayor, "promedio"])

grupo_mediana_mayor = estadisticas_validas["mediana"].idxmax()
mediana_mayor = float(estadisticas_validas.loc[grupo_mediana_mayor, "mediana"])

grupo_mayor_dispersion = estadisticas_validas["desvio_estandar"].idxmax()
dispersion_mayor = float(estadisticas_validas.loc[grupo_mayor_dispersion, "desvio_estandar"])

mediana_leve = estadisticas_validas.loc["LEVE", "mediana"] if "LEVE" in estadisticas_validas.index else pd.NA
mediana_mortal = estadisticas_validas.loc["MORTAL", "mediana"] if "MORTAL" in estadisticas_validas.index else pd.NA

insights_edad = [
    f"El grupo con mayor edad promedio es {grupo_promedio_mayor}, con un promedio de {promedio_mayor:.2f} anios.",
    f"El grupo con mayor mediana de edad es {grupo_mediana_mayor}, con una mediana de {mediana_mayor:.2f} anios.",
    f"La mayor dispersion de edad aparece en {grupo_mayor_dispersion}, con un desvio estandar de {dispersion_mayor:.2f} anios.",
]

if pd.notna(mediana_leve) and pd.notna(mediana_mortal):
    diferencia_mediana = float(mediana_mortal - mediana_leve)
    if diferencia_mediana > 0:
        insights_edad.append(
            f"La mediana de edad en MORTAL supera a la de LEVE por {diferencia_mediana:.2f} anios, lo que sugiere una posible vulnerabilidad etaria en los casos fatales."
        )
    elif diferencia_mediana < 0:
        insights_edad.append(
            f"La mediana de edad en MORTAL es menor que la de LEVE por {abs(diferencia_mediana):.2f} anios; este patron requiere revisar exposicion, modo de desplazamiento y volumen de casos."
        )
    else:
        insights_edad.append(
            "La mediana de edad en MORTAL y LEVE es igual en esta muestra, por lo que no aparece una diferencia central clara entre esos grupos."
        )

for insight in insights_edad:
    print(f"- {insight}")

### Interpretacion escrita

Si los grupos `GRAVE` o `MORTAL` muestran edades centrales mas altas, una interpretacion plausible es que la edad incremente la vulnerabilidad corporal ante un mismo evento vial. Sin embargo, el analisis no permite afirmar que la edad cause mayor gravedad. La severidad observada puede depender de multiples factores no controlados: tipo de usuario vial, velocidad, infraestructura, mecanismo del siniestro, uso de protecciones, estado de salud previo y acceso a atencion medica.

La diferencia entre correlacion y causalidad es clave. Una asociacion descriptiva entre edad y gravedad indica que ambas variables se mueven juntas en el dataset, pero no demuestra que una produzca la otra. Para avanzar hacia explicaciones causales se requeririan disenos analiticos adicionales, control de confundidores y posiblemente otras fuentes de datos.

Desde el storytelling con datos, este bloque ayuda a identificar posibles vulnerabilidades etarias y a formular nuevas preguntas: que edades aparecen con mayor frecuencia en lesiones graves o mortales?, esas edades se concentran en peatones, ciclistas o motociclistas?, la relacion se mantiene si se separa por modo de desplazamiento?

### 6. Limitaciones

- Se usan solo registros con `edad_victima` valida; la exclusion de `SD` o edades no numericas puede introducir sesgo si la falta de edad no es aleatoria.
- Las clases de gravedad estan desbalanceadas, por lo que las distribuciones de `GRAVE` y especialmente `MORTAL` pueden tener menor estabilidad.
- El analisis no controla por modo de desplazamiento, rol de la victima, ubicacion, horario ni exposicion al riesgo.
- Las visualizaciones muestran asociacion descriptiva, no causalidad.
- La edad registrada depende de la calidad del dato fuente y de las reglas institucionales de carga.

## Analisis 3: Rol de la victima vs gravedad

### 1. Pregunta analitica

Ciertos roles dentro del siniestro presentan mayor severidad relativa que otros?

Esta pregunta permite analizar la vulnerabilidad vial no solo por el modo de desplazamiento, sino por la posicion que ocupaba la persona en el hecho: conductor, pasajero, peaton, ciclista u otros roles definidos en el diccionario oficial.

### 2. Hipotesis

La hipotesis exploratoria es que los roles con mayor exposicion fisica directa al impacto, como peatones o ciclistas, podrian presentar una mayor proporcion relativa de casos `GRAVE` o `MORTAL`. En cambio, roles con mayor proteccion estructural o menor exposicion directa podrian concentrar mayor proporcion de lesiones leves.

El concepto de **vulnerabilidad vial** refiere a la mayor probabilidad de sufrir consecuencias severas ante un siniestro, no necesariamente a una mayor responsabilidad en el evento. Desde esta perspectiva, el rol de la victima ayuda a distinguir exposicion corporal, proteccion disponible y severidad potencial.

### 3. Tablas descriptivas

Se construye una tabla cruzada entre `rol_victima` y `gravedad_victima`, normalizada por fila. Cada fila representa el 100% de los registros de un rol y permite comparar la composicion relativa de gravedad entre roles.

In [ ]:
if "rol_victima" not in df_analisis.columns:
    raise KeyError("No se encontro la columna rol_victima en el dataset procesado.")

df_rol = df_analisis.copy()
df_rol["rol_victima"] = (
    df_rol["rol_victima"]
    .astype("string")
    .str.strip()
    .str.upper()
)

tabla_rol_frecuencias = pd.crosstab(
    df_rol["rol_victima"],
    df_rol["gravedad_victima"],
    dropna=False,
).reindex(columns=orden_gravedad, fill_value=0)

tabla_rol_frecuencias["TOTAL"] = tabla_rol_frecuencias.sum(axis=1)
tabla_rol_frecuencias = tabla_rol_frecuencias.sort_values("TOTAL", ascending=False)

tabla_rol_porcentual = (
    pd.crosstab(
        df_rol["rol_victima"],
        df_rol["gravedad_victima"],
        normalize="index",
        dropna=False,
    )
    .reindex(columns=orden_gravedad, fill_value=0)
    .mul(100)
    .round(2)
    .reindex(tabla_rol_frecuencias.index)
)

tabla_rol_porcentual["TOTAL_REGISTROS"] = tabla_rol_frecuencias["TOTAL"]

display(tabla_rol_porcentual)

#### Tabla resumen ordenada por porcentaje de casos graves/mortales

La siguiente tabla resume la severidad relativa de cada rol. El indicador `pct_grave_mortal` combina `GRAVE` y `MORTAL` para identificar roles con mayor proporcion de consecuencias severas.

In [ ]:
tabla_resumen_rol = tabla_rol_porcentual.copy()
tabla_resumen_rol["pct_grave_mortal"] = (
    tabla_resumen_rol[["GRAVE", "MORTAL"]].sum(axis=1).round(2)
)

tabla_resumen_rol = (
    tabla_resumen_rol
    .reset_index()
    .rename(columns={"rol_victima": "rol_victima"})
    .sort_values("pct_grave_mortal", ascending=False)
)

display(tabla_resumen_rol[["rol_victima", "TOTAL_REGISTROS", "LEVE", "GRAVE", "MORTAL", "pct_grave_mortal"]])

### 4. Visualizaciones

#### Grafico de barras apiladas porcentuales

El grafico apilado porcentual permite comparar la estructura de severidad dentro de cada rol. Todas las barras suman 100%, por lo que la lectura se centra en diferencias relativas y no en volumen absoluto.

In [ ]:
plot_rol_porcentual = tabla_rol_porcentual[orden_gravedad].copy()

ax = plot_rol_porcentual.plot(
    kind="bar",
    stacked=True,
    figsize=(11, 6),
    color=["#4C78A8", "#F58518", "#E45756"],
)

ax.set_title("Composicion porcentual de gravedad segun rol de la victima")
ax.set_xlabel("Rol de la victima")
ax.set_ylabel("Porcentaje dentro del rol")
ax.legend(title="Gravedad", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_ylim(0, 100)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

#### Heatmap porcentual

El heatmap facilita detectar rapidamente que roles presentan mayor peso relativo de lesiones graves o mortales. Los colores representan porcentajes dentro de cada rol, no cantidades absolutas.

In [ ]:
plt.figure(figsize=(8, max(4, 0.45 * len(tabla_rol_porcentual))))
sns.heatmap(
    tabla_rol_porcentual[orden_gravedad],
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.5,
    cbar_kws={"label": "Porcentaje dentro del rol"},
)
plt.title("Heatmap porcentual de gravedad por rol de la victima")
plt.xlabel("Gravedad de la victima")
plt.ylabel("Rol de la victima")
plt.tight_layout()
plt.show()

### 5. Insights escritos

La interpretacion automatica identifica roles con mayor proporcion de severidad alta. Para evitar errores conceptuales, los insights sustantivos excluyen `SD`, ya que significa **Sin Datos** y no representa un rol real dentro del siniestro.

In [ ]:
metricas_rol = tabla_rol_frecuencias.copy()
metricas_rol["pct_leve"] = tabla_rol_porcentual["LEVE"]
metricas_rol["pct_grave"] = tabla_rol_porcentual["GRAVE"]
metricas_rol["pct_mortal"] = tabla_rol_porcentual["MORTAL"]
metricas_rol["pct_grave_mortal"] = tabla_rol_porcentual[["GRAVE", "MORTAL"]].sum(axis=1).round(2)

incluye_sd_rol = SD_VALUE in metricas_rol.index
metricas_rol_sustantivas = metricas_rol.drop(index=SD_VALUE, errors="ignore")
metricas_rol_sustantivas = metricas_rol_sustantivas[
    metricas_rol_sustantivas.index.notna()
    & (metricas_rol_sustantivas.index.astype("string") != "<NA>")
]

roles_evaluables = metricas_rol_sustantivas[
    (metricas_rol_sustantivas["TOTAL"] >= 30)
    & (metricas_rol_sustantivas["LEVE"] > 0)
    & ((metricas_rol_sustantivas["GRAVE"] + metricas_rol_sustantivas["MORTAL"]) > 0)
]

roles_solo_mortales = metricas_rol_sustantivas[
    (metricas_rol_sustantivas["TOTAL"] >= 30)
    & (metricas_rol_sustantivas["pct_mortal"] == 100)
].index.tolist()

rol_mayor_volumen = metricas_rol["TOTAL"].idxmax()
total_rol_mayor_volumen = int(metricas_rol.loc[rol_mayor_volumen, "TOTAL"])

rol_sustantivo_mayor_volumen = metricas_rol_sustantivas["TOTAL"].idxmax()
total_rol_sustantivo_mayor_volumen = int(
    metricas_rol_sustantivas.loc[rol_sustantivo_mayor_volumen, "TOTAL"]
)

base_ranking_rol = roles_evaluables if not roles_evaluables.empty else metricas_rol_sustantivas

rol_mayor_grave_mortal = base_ranking_rol["pct_grave_mortal"].idxmax()
pct_rol_mayor_grave_mortal = float(
    base_ranking_rol.loc[rol_mayor_grave_mortal, "pct_grave_mortal"]
)

rol_mayor_mortal = base_ranking_rol["pct_mortal"].idxmax()
pct_rol_mayor_mortal = float(base_ranking_rol.loc[rol_mayor_mortal, "pct_mortal"])

rol_mayor_leve = base_ranking_rol["pct_leve"].idxmax()
pct_rol_mayor_leve = float(base_ranking_rol.loc[rol_mayor_leve, "pct_leve"])

roles_bajo_volumen = metricas_rol_sustantivas[metricas_rol_sustantivas["TOTAL"] < 30].index.tolist()

insights_rol = [
    f"Incluyendo todas las categorias, el rol con mayor volumen es {rol_mayor_volumen}, con {total_rol_mayor_volumen} registros.",
    f"Excluyendo {SD_VALUE} y faltantes, el rol sustantivo con mayor volumen es {rol_sustantivo_mayor_volumen}, con {total_rol_sustantivo_mayor_volumen} registros.",
    f"Entre roles evaluables, la mayor proporcion combinada de casos GRAVE o MORTAL aparece en {rol_mayor_grave_mortal}, con {pct_rol_mayor_grave_mortal:.2f}% dentro del rol.",
    f"Entre roles evaluables, la mayor proporcion de casos MORTAL aparece en {rol_mayor_mortal}, con {pct_rol_mayor_mortal:.2f}% dentro del rol.",
    f"El rol evaluable con mayor proporcion de casos LEVE es {rol_mayor_leve}, con {pct_rol_mayor_leve:.2f}% dentro del rol.",
]

if roles_solo_mortales:
    insights_rol.append(
        "Algunos roles aparecen con 100% de casos MORTAL, lo que puede indicar sesgo de completitud o registro diferenciado y no debe leerse automaticamente como mayor vulnerabilidad causal: "
        + ", ".join(roles_solo_mortales)
        + "."
    )

if roles_bajo_volumen:
    insights_rol.append(
        "Los siguientes roles sustantivos tienen bajo volumen de registros y sus porcentajes pueden ser inestables: "
        + ", ".join(roles_bajo_volumen)
        + "."
    )

if incluye_sd_rol:
    insights_rol.append(
        f"La categoria {SD_VALUE} debe interpretarse como {SD_MEANING}; informa calidad de datos y no un rol vial sustantivo."
    )

for insight in insights_rol:
    print(f"- {insight}")

### Interpretacion escrita

Si roles como `PEATON` o `CICLISTA` presentan mayor proporcion relativa de casos graves o mortales, una interpretacion plausible es la mayor exposicion fisica al impacto. A diferencia de ocupantes protegidos por una estructura vehicular, estos roles pueden absorber directamente la energia del siniestro, lo que incrementa la probabilidad de consecuencias severas.

La vulnerabilidad vial no debe entenderse como responsabilidad individual. Un rol vulnerable es aquel que, ante una interaccion vial adversa, tiene menor proteccion material y mayor probabilidad de sufrir lesiones severas. Por eso, el analisis de rol es relevante para pensar prevencion, infraestructura segura y priorizacion de politicas publicas.

Las diferencias entre categorias tambien pueden reflejar exposicion diferencial: algunos roles circulan mas en determinadas zonas, horarios o modos de desplazamiento. Por lo tanto, el patron observado debe interpretarse como una senal descriptiva y no como una explicacion completa.

### 6. Limitaciones

- El analisis es descriptivo y no permite inferir causalidad.
- `SD` representa ausencia de informacion y no debe compararse como rol real.
- Las proporciones pueden ser inestables en roles con pocos registros.
- No se controla por edad, modo de desplazamiento, zona, horario, velocidad, infraestructura ni exposicion al riesgo.
- Algunas categorias pueden solaparse conceptualmente con modo de desplazamiento, por ejemplo `CICLISTA` o `PEATON`, por lo que conviene revisar consistencia entre variables antes de conclusiones definitivas.
- La severidad observada depende de la calidad de registro y de las definiciones institucionales del dataset.

## Analisis 4: Evolucion temporal de la gravedad de los siniestros

### 1. Pregunta analitica

Existen cambios temporales en la distribucion de victimas leves, graves y mortales a lo largo de los a?os registrados?

El analisis temporal es importante porque permite observar si la severidad se mantiene estable, aumenta, disminuye o cambia su composicion relativa. En seguridad vial, mirar la evolucion anual ayuda a construir una narrativa sobre persistencias y rupturas, aunque no permite explicar por si solo las causas de esos cambios.

### 2. Hipotesis

La hipotesis exploratoria es que la distribucion de gravedad puede variar entre a?os por cambios en movilidad, infraestructura, politicas publicas, comportamiento vial, registro institucional o factores externos no presentes en el dataset. Se espera que las victimas leves dominen el volumen, pero los casos `GRAVE` y `MORTAL` son especialmente relevantes por su impacto sanitario y social.

### 3. Tablas descriptivas

Se construyen dos tablas: una con la cantidad de victimas por a?o y gravedad, y otra con la distribucion porcentual de gravedad dentro de cada a?o. La primera mide volumen; la segunda permite comparar composicion anual sin que el tama?o de cada a?o domine la interpretacion.

In [ ]:
if "anio_siniestro" not in df_analisis.columns:
    raise KeyError("No se encontro la columna anio_siniestro en el dataset procesado.")

df_tiempo = df_analisis.copy()
df_tiempo["anio_siniestro"] = pd.to_numeric(df_tiempo["anio_siniestro"], errors="coerce")
df_tiempo = df_tiempo.dropna(subset=["anio_siniestro", "gravedad_victima"]).copy()
df_tiempo["anio_siniestro"] = df_tiempo["anio_siniestro"].astype(int)

tabla_anual_frecuencias = pd.crosstab(
    df_tiempo["anio_siniestro"],
    df_tiempo["gravedad_victima"],
    dropna=False,
).reindex(columns=orden_gravedad, fill_value=0)

tabla_anual_frecuencias["TOTAL"] = tabla_anual_frecuencias.sum(axis=1)
tabla_anual_frecuencias = tabla_anual_frecuencias.sort_index()

print("Cantidad de victimas por a?o y gravedad")
display(tabla_anual_frecuencias)

tabla_anual_porcentual = (
    tabla_anual_frecuencias[orden_gravedad]
    .div(tabla_anual_frecuencias["TOTAL"].replace(0, pd.NA), axis=0)
    .mul(100)
    .round(2)
)
tabla_anual_porcentual["TOTAL_REGISTROS"] = tabla_anual_frecuencias["TOTAL"]

print("Distribucion porcentual de gravedad por a?o")
display(tabla_anual_porcentual)

### 4. Visualizaciones

#### Lineplot: cantidad de victimas por a?o

Este grafico muestra la evolucion del volumen total de victimas registradas por a?o. Sirve para detectar aumentos, disminuciones o posibles quiebres temporales en la serie.

In [ ]:
serie_total_anual = (
    tabla_anual_frecuencias["TOTAL"]
    .rename("cantidad_victimas")
    .reset_index()
)

plt.figure(figsize=(9, 5))
sns.lineplot(
    data=serie_total_anual,
    x="anio_siniestro",
    y="cantidad_victimas",
    marker="o",
    linewidth=2,
    color="#4C78A8",
)
plt.title("Cantidad de victimas por a?o")
plt.xlabel("A?o del siniestro")
plt.ylabel("Cantidad de victimas")
plt.xticks(serie_total_anual["anio_siniestro"].tolist())
plt.tight_layout()
plt.show()

#### Lineplot: evolucion de casos graves y mortales

Los casos `GRAVE` y `MORTAL` se observan por separado y tambien combinados como severidad alta. Esta lectura permite poner el foco en los eventos de mayor impacto, aun cuando sean menos frecuentes que los casos leves.

In [ ]:
serie_grave_mortal = tabla_anual_frecuencias[["GRAVE", "MORTAL"]].copy()
serie_grave_mortal["GRAVE_MORTAL"] = serie_grave_mortal["GRAVE"] + serie_grave_mortal["MORTAL"]
serie_grave_mortal = (
    serie_grave_mortal
    .reset_index()
    .melt(
        id_vars="anio_siniestro",
        value_vars=["GRAVE", "MORTAL", "GRAVE_MORTAL"],
        var_name="tipo_severidad",
        value_name="cantidad",
    )
)

plt.figure(figsize=(9, 5))
sns.lineplot(
    data=serie_grave_mortal,
    x="anio_siniestro",
    y="cantidad",
    hue="tipo_severidad",
    marker="o",
    linewidth=2,
    palette={"GRAVE": "#F58518", "MORTAL": "#E45756", "GRAVE_MORTAL": "#6F4E7C"},
)
plt.title("Evolucion anual de casos graves y mortales")
plt.xlabel("A?o del siniestro")
plt.ylabel("Cantidad de victimas")
plt.xticks(sorted(df_tiempo["anio_siniestro"].unique()))
plt.tight_layout()
plt.show()

#### Grafico apilado porcentual: gravedad por a?o

El grafico apilado porcentual permite observar si la composicion de gravedad cambia con el tiempo. Cada barra representa el 100% de las victimas de un a?o.

In [ ]:
plot_anual_porcentual = tabla_anual_porcentual[orden_gravedad].copy()

ax = plot_anual_porcentual.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6),
    color=["#4C78A8", "#F58518", "#E45756"],
)

ax.set_title("Distribucion porcentual de gravedad por a?o")
ax.set_xlabel("A?o del siniestro")
ax.set_ylabel("Porcentaje dentro del a?o")
ax.legend(title="Gravedad", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_ylim(0, 100)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 5. Insights escritos

La interpretacion automatica identifica tendencias simples entre el primer y ultimo a?o disponible, maximos y minimos anuales, y cambios en la severidad alta (`GRAVE + MORTAL`). Se trata de una lectura descriptiva, no de una estimacion causal.

In [ ]:
primer_anio = int(tabla_anual_frecuencias.index.min())
ultimo_anio = int(tabla_anual_frecuencias.index.max())

total_primer_anio = int(tabla_anual_frecuencias.loc[primer_anio, "TOTAL"])
total_ultimo_anio = int(tabla_anual_frecuencias.loc[ultimo_anio, "TOTAL"])
delta_total = total_ultimo_anio - total_primer_anio
pct_delta_total = (delta_total / total_primer_anio * 100) if total_primer_anio else 0

anio_max_total = int(tabla_anual_frecuencias["TOTAL"].idxmax())
max_total = int(tabla_anual_frecuencias.loc[anio_max_total, "TOTAL"])
anio_min_total = int(tabla_anual_frecuencias["TOTAL"].idxmin())
min_total = int(tabla_anual_frecuencias.loc[anio_min_total, "TOTAL"])

severidad_alta_anual = tabla_anual_frecuencias["GRAVE"] + tabla_anual_frecuencias["MORTAL"]
pct_severidad_alta_anual = tabla_anual_porcentual["GRAVE"] + tabla_anual_porcentual["MORTAL"]

anio_max_severidad_alta = int(severidad_alta_anual.idxmax())
max_severidad_alta = int(severidad_alta_anual.loc[anio_max_severidad_alta])

anio_max_pct_severidad_alta = int(pct_severidad_alta_anual.idxmax())
max_pct_severidad_alta = float(pct_severidad_alta_anual.loc[anio_max_pct_severidad_alta])

pct_alta_primer_anio = float(pct_severidad_alta_anual.loc[primer_anio])
pct_alta_ultimo_anio = float(pct_severidad_alta_anual.loc[ultimo_anio])
delta_pct_alta = pct_alta_ultimo_anio - pct_alta_primer_anio

if delta_total > 0:
    tendencia_total = "aumento"
elif delta_total < 0:
    tendencia_total = "disminucion"
else:
    tendencia_total = "estabilidad"

if delta_pct_alta > 0:
    tendencia_severidad = "aumento relativo"
elif delta_pct_alta < 0:
    tendencia_severidad = "disminucion relativa"
else:
    tendencia_severidad = "estabilidad relativa"

insights_temporales = [
    f"Entre {primer_anio} y {ultimo_anio}, el total anual de victimas muestra {tendencia_total}: pasa de {total_primer_anio} a {total_ultimo_anio} registros ({pct_delta_total:.2f}%).",
    f"El a?o con mayor cantidad total de victimas es {anio_max_total}, con {max_total} registros; el menor volumen aparece en {anio_min_total}, con {min_total} registros.",
    f"El mayor volumen absoluto de casos GRAVE + MORTAL aparece en {anio_max_severidad_alta}, con {max_severidad_alta} registros.",
    f"La mayor proporcion anual de GRAVE + MORTAL aparece en {anio_max_pct_severidad_alta}, con {max_pct_severidad_alta:.2f}% del total de ese a?o.",
    f"Entre {primer_anio} y {ultimo_anio}, la proporcion de GRAVE + MORTAL muestra {tendencia_severidad}: cambia de {pct_alta_primer_anio:.2f}% a {pct_alta_ultimo_anio:.2f}%.",
]

for insight in insights_temporales:
    print(f"- {insight}")

### Interpretacion escrita

La evolucion temporal permite distinguir cambios en volumen y cambios en composicion. Un aumento en la cantidad total de victimas puede responder a mayor circulacion, cambios en registro, variaciones demograficas, condiciones economicas, politicas de movilidad o eventos externos. A su vez, una modificacion en la proporcion de casos graves o mortales puede sugerir cambios en la severidad relativa, pero no explica por si misma sus causas.

Desde un enfoque de storytelling, la pregunta central no es solo si hay mas o menos victimas, sino si el perfil de gravedad cambia con el tiempo. Si la proporcion de `GRAVE + MORTAL` aumenta, podria indicar una se?al de alerta para profundizar. Si disminuye, podria sugerir mejoras relativas o cambios en composicion de registros. En ambos casos, la interpretacion requiere contexto externo.

### 6. Limitaciones

- El analisis temporal es descriptivo y no permite atribuir causalidad.
- No se incorporan factores externos como cambios normativos, infraestructura, niveles de movilidad, pandemia, controles viales, clima o transformaciones urbanas.
- Las variaciones anuales pueden reflejar cambios en la calidad o criterio de registro, no solo cambios reales en siniestralidad.
- No se ajusta por poblacion, cantidad de viajes, parque vehicular ni exposicion al riesgo.
- La serie anual puede ser corta para detectar tendencias robustas.
- La comparacion de porcentajes debe interpretarse junto con los conteos absolutos para evitar conclusiones basadas en denominadores peque?os.

## 7. Conclusiones parciales

El cruce entre gravedad de la victima y modo de desplazamiento permite construir una primera narrativa analitica sobre la severidad de los siniestros viales. La principal contribucion del analisis no es solo mostrar que categorias tienen mas casos, sino distinguir volumen absoluto de severidad relativa.

Como conclusion parcial, los patrones observados deben leerse como evidencia exploratoria. Sirven para priorizar preguntas, orientar visualizaciones posteriores y detectar posibles focos de vulnerabilidad, pero requieren ser complementados con variables contextuales y controles adicionales. En esta etapa, el aporte central es consolidar una lectura reproducible, semanticamente informada por el diccionario oficial y comunicable en terminos de insights.

## Conclusiones generales

### 1. Hallazgos principales

Los analisis realizados muestran que la gravedad de los siniestros viales no se distribuye de manera homogenea entre categorias. En el cruce entre gravedad y modo de desplazamiento, la categoria `SD` concentra un volumen importante de registros, lo que obliga a separar calidad de datos de interpretacion sustantiva. Al excluir `SD`, `MOTO` aparece como el modo sustantivo con mayor volumen de victimas, mientras que `PEATON` presenta la mayor proporcion relativa de casos `GRAVE + MORTAL`.

El analisis de edad sugiere una posible relacion entre mayor edad y mayor severidad. En los registros con edad valida, el grupo `MORTAL` presenta la mayor edad promedio y la mayor mediana. Esta diferencia no debe leerse como causalidad directa, pero si como una senal relevante de vulnerabilidad etaria que merece ser profundizada.

En el analisis por rol, `SD` vuelve a aparecer como una categoria dominante en volumen, reforzando la importancia de la completitud del registro. Entre los roles sustantivos evaluables, `PEATON` muestra la mayor proporcion de casos graves o mortales. Tambien se observaron categorias como `CONDUCTOR` y `PASAJERO` con 100% de casos mortales, lo que probablemente refleja sesgos o diferencias de completitud en la forma de registro y no debe interpretarse automaticamente como mayor riesgo causal.

La evolucion temporal muestra que entre 2019 y 2024 el total anual de victimas aumenta levemente y que el menor volumen se observa en 2020, un a?o que requiere interpretacion contextual por posibles cambios externos en la movilidad. La proporcion anual de `GRAVE + MORTAL` tambien muestra un incremento relativo entre el primer y el ultimo a?o disponible, aunque la serie es corta y descriptiva.

### 2. Insights urbanos/sociales

Desde una perspectiva urbana y social, los resultados sugieren que la vulnerabilidad vial esta asociada a condiciones de exposicion fisica. Peatones y otros usuarios con menor proteccion estructural aparecen como grupos de interes para el analisis de severidad. Esto no implica responsabilidad individual, sino una mayor fragilidad ante el impacto y una necesidad de mirar la infraestructura, la convivencia vial y las condiciones de circulacion.

El peso de `SD` en variables clave tambien es un hallazgo social y metodologico. La ausencia de datos no es un detalle tecnico menor: condiciona que grupos pueden ser interpretados con precision y cuales quedan parcialmente invisibilizados. En una problematica de seguridad vial, la calidad del registro forma parte de la calidad de la respuesta publica posible.

La edad agrega otra dimension de vulnerabilidad. Si las victimas mortales presentan edades centrales mas altas, el analisis invita a considerar politicas y entornos urbanos sensibles a personas mayores: tiempos de cruce, accesibilidad peatonal, velocidades permitidas, visibilidad, senalizacion y proteccion en intersecciones.

### 3. Interpretaciones generales

El principal aporte del notebook es pasar de un EDA tecnico a una narrativa analitica integrada. Los datos muestran volumen, composicion y evolucion; la interpretacion convierte esos patrones en preguntas relevantes para la seguridad vial. En conjunto, los resultados apuntan a tres ejes: vulnerabilidad por modo de desplazamiento, vulnerabilidad por rol y vulnerabilidad etaria.

La variable `gravedad_victima` debe mantenerse como variable ordinal. Tratar `LEVE`, `GRAVE` y `MORTAL` como textos independientes debilitaria la lectura del problema, porque la severidad tiene una jerarquia institucional clara. Del mismo modo, `SD` debe entenderse como **Sin Datos** y no como categoria real del fenomeno.

Para una presentacion oral o defensa del TP, una idea central seria: los siniestros viales no solo deben analizarse por cantidad, sino por severidad y por grupos de victimas. La misma cantidad de registros puede tener implicancias muy distintas si cambia la proporcion de casos graves o mortales, o si esos casos se concentran en usuarios mas vulnerables.

### 4. Limitaciones del analisis

Este notebook desarrolla analisis exploratorio avanzado, no inferencia causal. Las asociaciones observadas no prueban que una variable produzca mayor gravedad. Para sostener explicaciones causales seria necesario controlar por factores de confusion y complementar el dataset con informacion externa.

Entre las principales limitaciones se encuentran:

- alta presencia de `SD` en variables relevantes;
- desbalanceo de clases, especialmente por la baja frecuencia relativa de `MORTAL`;
- ausencia de variables contextuales como velocidad, tipo de via, comuna, clima, infraestructura, flujo vehicular o cantidad de viajes;
- posible diferencia en criterios de registro entre victimas leves, graves y mortales;
- serie temporal limitada para evaluar tendencias robustas;
- imposibilidad de medir exposicion real al riesgo solo con cantidad de victimas registradas.

Estas limitaciones no invalidan el analisis, pero definen su alcance: los resultados deben leerse como evidencia descriptiva y como base para preguntas posteriores.

### 5. Posibles mejoras futuras

Como siguientes pasos, el proyecto podria fortalecer el analisis incorporando nuevas dimensiones y controles. Una mejora importante seria enriquecer el dataset con variables territoriales, temporales y de infraestructura para evaluar si los patrones de gravedad se concentran en zonas, horarios o tipos de via especificos.

Tambien seria valioso profundizar el tratamiento de `SD`, distinguiendo entre faltantes recuperables, categorias no informadas y posibles sesgos sistematicos de carga. Esto permitiria mejorar la calidad analitica antes de avanzar hacia modelos predictivos.

Futuras lineas de trabajo podrian incluir:

- analisis por comuna, barrio o tipo de via;
- cruces entre edad, rol y modo de desplazamiento;
- comparacion entre frecuencia absoluta y tasas ajustadas por exposicion;
- analisis temporal mensual o estacional si la granularidad lo permite;
- evaluacion de outliers y consistencia de categorias;
- preparacion de variables para una etapa posterior de modelado, sin introducir modelos antes de cerrar la etapa exploratoria.

En sintesis, el notebook deja planteada una narrativa clara para el informe ejecutivo: la severidad vial debe analizarse con conocimiento de dominio, separando volumen de riesgo relativo, respetando la ordinalidad de la gravedad y reconociendo que la calidad de los datos condiciona la calidad de las conclusiones.